# Elliptic Curve Cryptography: From Theory to Bitcoin

**Source material:** *Elliptic Curve Cryptography research paper* (695.744 Reverse Engineering & Vulnerability Analysis)  
**Related:** [The Wright Trick (2016)](./A1-wright-trick.ipynb) | [Nonsense Signature (2018)](./A2-nonsense-signature.ipynb) | [Original Paper Notebook](./A3-original-paper.ipynb)

---

## Start at the End: What Problem Does ECC Solve?

In Bitcoin, ECC solves two problems:

**Problem 1 — the alias.** The ledger is public. Everyone can see every
transaction. You need people to be able to send you money, but you can't hand
out your private key — anyone who sees it could spend your funds. So you need
a **public key**: an alias derived from your private key that anyone can send
to, but that reveals nothing about the key itself. That's the one-way function:
$P = d \times G$ — easy to compute, impossible to reverse.

In Bitcoin this is literal. When someone sends you bitcoin, the transaction
locks the funds with a script that says: *"only someone who can produce a
public key that hashes to this value, AND a valid signature from that key, can
spend this."* That's **P2PKH** (Pay-to-Public-Key-Hash) — the alias isn't even
the public key directly, it's a hash of it. You only reveal $P$ when you spend,
along with the signature. The current standard, **Taproot** (P2TR), locks to
the public key directly and uses Schnorr signatures instead of ECDSA — but the
two-problem structure is the same.

**Problem 2 — the proof.** When you want to spend, you need to prove you
control the alias — that you know the private key behind it — without revealing
it. You can't show a password (the ledger is public, everyone would see it).
You can't rely on a trusted third party (there isn't one). You need a proof
that only you could produce, that anyone can verify, that reveals nothing about
your secret, and that can't be replayed. **That's a digital signature.**

ECC gives us both: the one-way function for the alias, and the algebraic
structure for the signature. We'll build ECDSA first (Module 5), then see how
Schnorr improves on it and why Taproot switched (Module 6). Before ECC, we already had answers:
RSA (1977) and ElGamal (1985) both work. But they need enormous keys — 3072 bits
for 128-bit security. That's slow, wasteful, and impractical for constrained devices.

In 1985, Neal Koblitz and Victor Miller independently asked: **what if we ran the
same discrete logarithm trick, but on a different mathematical structure — one where
the problem is fundamentally harder?**

The structure they found was **elliptic curves over finite fields.** Same security,
a fraction of the key size:

| Security | RSA / ElGamal | ECC | Savings |
|----------|--------------|-----|---------|
| 80-bit   | 1024 bits    | 160 bits | 6x smaller |
| 128-bit  | 3072 bits    | 256 bits | 12x smaller |
| 256-bit  | 15360 bits   | 512 bits | 30x smaller |

### What we need from the math

To solve both problems, we need:

**For the alias (Problem 1):**

- A **one-way function** — easy to compute forward, infeasible to reverse
- ECC answer: $P = d \times G$ (scalar multiplication on a curve)
- Forward: milliseconds. Reverse: more energy than the solar system contains.

**For the signature (Problem 2):**

- A **signing operation** — only the secret holder can produce it
  - ECDSA: $s = k^{-1}(z + r \cdot d) \bmod N$
  - Requires the private key $d$. No other way to produce a valid $(r, s)$.
  - Uses the one-way function *again*: a fresh random $k \to R = k \times G$ becomes part of the signature.
- A **public verification** — anyone can check it without learning the secret
  - ECDSA: check if $R'_x = r$ where $R' = (z/s)G + (r/s)P$
  - Uses only the public key $P$, the message hash $z$, and the signature $(r, s)$.

### But WHY does this work?

It works because elliptic curves over finite fields give us a one-way function
where "multiplication" means something geometrically elegant — drawing lines
through curve points, finding intersections, reflecting. And because the curve
points form an **algebraic group**, all the familiar rules of arithmetic hold,
but reversing the operation (the **discrete logarithm**) is computationally
infeasible.

To build this one-way function, we need a very specific mathematical environment —
one where numbers wrap around, where parallel lines meet at infinity, and where
points on a curve obey group axioms.

**That's the journey of this notebook:** starting from what we need to accomplish,
we build the mathematical machinery piece by piece, and see it come alive in
working code.

---

## How to Read This

Each module answers a question that the previous module raised:

```
"We need a digital signature scheme with small keys"
  └─→ "We need a one-way function on a curve"
        └─→ Module 4: The Discrete Log Problem — what makes it hard?
              └─→ Module 3: Point Operations — how do we "add" and "multiply"?
                    └─→ Module 2: Elliptic Curves — what are these curves?
                          └─→ Module 1: Algebraic Foundations — what rules govern this world?

Then we build back up with the answer:
  Module 5: ECDSA — the signature scheme we set out to build
  Module 6: Bitcoin Applications — where it all lands (Schnorr, onion routing)
  Module 7: Exercises — test your understanding
```

If you prefer, jump straight to Module 3 (the code) and circle back to
Modules 1-2 (the theory) when you want to understand WHY the formulas work.

